# Ontology ablation — development 6000

This notebook follows the same structure and basic ontology code as `03_CLIP_AST_ontology_6000`. Only two one-factor-at-a-time ablation switches are added: temporal persistence and the CLIP protection gate.


## Setup and frozen parameters


In [1]:
%run ../Acoustic_ontology.ipynb


Ontology rules: 22
Ontology event mappings: 22


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Project paths
PROJECT_ROOT = Path(r"C:\Users\22313906t\Desktop\TAU_project")
CLIP_PATH = PROJECT_ROOT / 'results_CLIP' / 'tau_6000_clip_predictions.csv'
AST_PATH = PROJECT_ROOT / 'results_AST_6000' / 'tau_all_ast_events.csv'

OUTPUT_DIR = PROJECT_ROOT / 'results_ONTOLOGY' / 'ablations_persistence_clip_gate'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUTPUT_DIR / 'tau_6000_persistence_clip_gate_ablation_summary.csv'
CONFIG_PATH = OUTPUT_DIR / 'tau_6000_persistence_clip_gate_ablation_config.json'

# Dataset settings
TAU_CLASSES = [
    'airport',
    'bus',
    'metro',
    'metro station',
    'park',
    'public square',
    'shopping mall',
    'pedestrian street',
    'street traffic',
    'tram',
]
EXPECTED_TOTAL = 6000
N_PER_CLASS = 600
CLIP_SCORE_COLUMNS = {scene: scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES}

# Basic-ontology parameters
MIN_EVENT_CONFIDENCE = 0.08
ONTOLOGY_LAMBDA = 0.4
CLIP_PROTECTION_MARGIN = 0.2
MIN_ONTOLOGY_EVIDENCE = 0.15
MIN_ONTOLOGY_ADVANTAGE = 0.05
STRONG_ONTOLOGY_EVIDENCE = 0.6

# Same active modes as Stage 03
NORMALIZATION_MODE = 'sum'
CLIP_PROTECTION_MODE = 'candidate'

print('=' * 78)
print('TAU 6000 DEVELOPMENT — ONTOLOGY ABLATION')
print('=' * 78)
print('\nCLIP:')
print(CLIP_PATH)
print('\nAST:')
print(AST_PATH)
print('\nOutput:')
print(OUTPUT_DIR)
print('\nOntology normalization:', NORMALIZATION_MODE.upper())
print('CLIP protection mode:', CLIP_PROTECTION_MODE)


TAU 6000 DEVELOPMENT — ONTOLOGY ABLATION

CLIP:
C:\Users\22313906t\Desktop\TAU_project\results_CLIP\tau_6000_clip_predictions.csv

AST:
C:\Users\22313906t\Desktop\TAU_project\results_AST_6000\tau_all_ast_events.csv

Output:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY\ablations_persistence_clip_gate

Ontology normalization: SUM
CLIP protection mode: candidate


## Load and prepare CLIP/AST results


In [3]:
# Load CLIP results
if not CLIP_PATH.exists():
    raise FileNotFoundError(f'Cannot find CLIP predictions:\n{CLIP_PATH}')
if not AST_PATH.exists():
    raise FileNotFoundError(f'Cannot find AST events:\n{AST_PATH}')
clip_df = pd.read_csv(CLIP_PATH)
required_clip_columns = {'clip_id', 'ground_truth', 'clip_top1', 'clip_top2', 'clip_top3'}
missing_clip_columns = required_clip_columns - set(clip_df.columns)
if missing_clip_columns:
    raise ValueError(f'Missing CLIP columns: {missing_clip_columns}')
clip_df['clip_id'] = clip_df['clip_id'].astype(str)
clip_df['ground_truth'] = clip_df['ground_truth'].astype(str)
clip_df = clip_df.drop_duplicates(subset=['clip_id'], keep='last').reset_index(drop=True)
for scene, column in CLIP_SCORE_COLUMNS.items():
    if column not in clip_df.columns:
        raise ValueError(f'Missing CLIP score column: {column}')

# Validate the dataset
print('\n' + '=' * 78)
print('CLIP DATASET CHECK')
print('=' * 78)
print('CLIP rows:', len(clip_df))
print('Unique IDs:', clip_df['clip_id'].nunique())
if len(clip_df) != EXPECTED_TOTAL:
    raise RuntimeError(f'Expected {EXPECTED_TOTAL} CLIP rows, found {len(clip_df)}.')
class_counts = clip_df['ground_truth'].value_counts().reindex(TAU_CLASSES, fill_value=0)
print('\nClass distribution:')
print(class_counts)
if not (class_counts == N_PER_CLASS).all():
    raise RuntimeError(f'Expected {N_PER_CLASS} clips per class.')

# Load AST results
ast_df = pd.read_csv(AST_PATH)
if 'clip_id' not in ast_df.columns:
    raise ValueError('AST file has no clip_id column.')
ast_df['clip_id'] = ast_df['clip_id'].astype(str)

# Detect AST event and score columns
EVENT_COL = None
for candidate in ['label', 'event', 'event_label', 'audio_event', 'class_name']:
    if candidate in ast_df.columns:
        EVENT_COL = candidate
        break
if EVENT_COL is None:
    raise ValueError(f'Cannot find AST event column. Available columns: {list(ast_df.columns)}')
SCORE_COL = None
for candidate in ['max_score', 'score', 'mean_score']:
    if candidate in ast_df.columns:
        SCORE_COL = candidate
        break
if SCORE_COL is None:
    raise ValueError(f'Cannot find AST score column. Available columns: {list(ast_df.columns)}')
print('\nAST event column:', EVENT_COL)
print('AST score column:', SCORE_COL)

# Persistence handling
# old behavior:
# use an existing persistence column if present;
# otherwise compute count / num_chunks;
# if neither is available, stop instead of silently using 1.0.
if 'persistence' not in ast_df.columns:
    if 'count' in ast_df.columns and 'num_chunks' in ast_df.columns:
        ast_df[
            'persistence'] = ast_df['count'].astype(float) / ast_df['num_chunks'].replace(0,
            np.nan).astype(float,
        )
        ast_df['persistence'] = ast_df['persistence'].fillna(0.0)
    else:
        raise ValueError('AST file needs either persistence or count + num_chunks.')

# Filter AST to CLIP ids
clip_ids = set(clip_df['clip_id'])
ast_df = ast_df[ast_df['clip_id'].isin(clip_ids)].copy()
ast_matched_ids = set(ast_df['clip_id'])
missing_ast_ids = clip_ids - ast_matched_ids
print('\n' + '=' * 78)
print('AST / CLIP MATCH CHECK')
print('=' * 78)
print('CLIP clips:', len(clip_ids))
print('AST matched clips:', len(ast_matched_ids))
print('Missing AST clips:', len(missing_ast_ids))
if missing_ast_ids:
    print('\nFirst missing AST IDs:')
    for clip_id in sorted(missing_ast_ids)[:20]:
        print(clip_id)
    raise RuntimeError('AST results do not cover all CLIP clips.')

# Normalize AST event names and numeric fields
ast_df['event_norm'] = ast_df[EVENT_COL].apply(normalize_event)
ast_df['event_score'] = pd.to_numeric(ast_df[SCORE_COL], errors='coerce').fillna(0.0)
ast_df['persistence'] = pd.to_numeric(ast_df['persistence'], errors='coerce').fillna(0.0)

# Index AST by CLIP
ast_groups = {clip_id: group.copy() for clip_id, group in ast_df.groupby('clip_id')}



CLIP DATASET CHECK
CLIP rows: 6000
Unique IDs: 6000

Class distribution:
ground_truth
airport              600
bus                  600
metro                600
metro station        600
park                 600
public square        600
shopping mall        600
pedestrian street    600
street traffic       600
tram                 600
Name: count, dtype: int64

AST event column: label
AST score column: max_score

AST / CLIP MATCH CHECK
CLIP clips: 6000
AST matched clips: 6000
Missing AST clips: 0


## Helper functions


In [4]:
# Safe float
def safe_float(value, default=0.0):
    try:
        value = float(value)
        if np.isfinite(value):
            return value
    except Exception:
        pass
    return default

# Ontology normalization
# "sum" = historical old-code behavior
# "max" = optional comparison
def normalize_ontology_scores(ontology_scores, candidates, mode='sum'):
    if mode == 'sum':
        denominator = sum((ontology_scores[scene] for scene in candidates))
    elif mode == 'max':
        denominator = max((ontology_scores[scene] for scene in candidates))
    else:
        raise ValueError("normalization mode must be 'sum' or 'max'")
    if denominator > 0:
        return {scene: ontology_scores[scene] / denominator for scene in candidates}
    return {scene: 0.0 for scene in candidates}

# CLIP protection gap
# "candidate" = historical old-code behavior:
# Top-1 CLIP score - ontology-proposed candidate CLIP score
# "top2" = optional newer comparison:
# Top-1 CLIP score - Top-2 CLIP score
def compute_clip_protection_gap(clip_scores, candidates, proposed_winner, mode='candidate'):
    original_top1 = candidates[0]
    original_score = clip_scores[original_top1]
    if mode == 'candidate':
        comparison_scene = proposed_winner
    elif mode == 'top2':
        if len(candidates) >= 2:
            comparison_scene = candidates[1]
        else:
            comparison_scene = original_top1
    else:
        raise ValueError("CLIP protection mode must be 'candidate' or 'top2'")
    comparison_score = clip_scores[comparison_scene]
    visual_gap = original_score - comparison_score
    return max(0.0, visual_gap)


## Ontology reasoning with ablation switches


In [5]:
# Run basic ontology for one CLIP
def run_ontology_for_clip(
    clip_row,
    audio_group,
    normalization_mode='sum',
    clip_protection_mode='candidate',
    use_persistence=True,
    use_clip_gate=True,
):
    candidates = [str(clip_row['clip_top1']), str(clip_row['clip_top2']), str(clip_row['clip_top3'])]
    candidates = list(dict.fromkeys(candidates))
    original_top1 = candidates[0]
    clip_scores = {scene: safe_float(clip_row[CLIP_SCORE_COLUMNS[scene]]) for scene in candidates}
    ontology_scores = {scene: 0.0 for scene in candidates}
    support_scores = {scene: 0.0 for scene in candidates}
    refine_scores = {scene: 0.0 for scene in candidates}
    fired_rules = []
    ignored_events = []
    evidence_rows = []
    if audio_group is not None:
        for _, event_row in audio_group.iterrows():
            event = event_row['event_norm']
            confidence = safe_float(event_row['event_score'])
            persistence = safe_float(event_row['persistence']) if use_persistence else 1.0
            if confidence < MIN_EVENT_CONFIDENCE:
                continue
            rule = RULE_LOOKUP.get(event)
            if rule is None:
                continue
            if rule['relation'] == 'IGNORE':
                ignored_events.append(f'{event}({confidence:.3f})')
                continue
            relation = rule['relation']
            for target_scene, semantic_strength in rule['targets'].items():
                if target_scene not in candidates:
                    continue
                evidence = confidence * persistence * semantic_strength
                ontology_scores[target_scene] += evidence
                if relation == 'SUPPORT':
                    support_scores[target_scene] += evidence
                elif relation == 'REFINE':
                    refine_scores[target_scene] += evidence
                fired_rules.append(f"{rule['name']}:{event}->{target_scene}[{relation};conf={confidence:.3f};pers={persistence:.2f};sem={semantic_strength:.2f};E={evidence:.3f}]")
                evidence_rows.append({'clip_id': str(clip_row['clip_id']), 'event': event_row[EVENT_COL], 'event_norm': event, 'event_max_score': confidence, 'persistence': persistence, 'ontology_rule': rule['name'], 'relation': relation, 'target_scene': target_scene, 'semantic_strength': semantic_strength, 'evidence': evidence})
    ontology_normalized = normalize_ontology_scores(
        ontology_scores=ontology_scores,
        candidates=candidates,
        mode=normalization_mode,
    )
    adjusted_scores = {scene: clip_scores[scene] + ONTOLOGY_LAMBDA * ontology_normalized[scene] for scene in candidates}
    proposed_winner = max(candidates, key=lambda scene: adjusted_scores[scene])
    clip_protection_gap = compute_clip_protection_gap(
        clip_scores=clip_scores,
        candidates=candidates,
        proposed_winner=proposed_winner,
        mode=clip_protection_mode,
    )
    current_ontology = ontology_scores[original_top1]
    alternative_ontology = ontology_scores[proposed_winner]
    ontology_advantage = alternative_ontology - current_ontology
    final_prediction = original_top1
    decision = 'KEEP_CLIP'
    reason = 'CLIP Top-1 retained'
    if proposed_winner != original_top1:
        enough_ontology = alternative_ontology >= MIN_ONTOLOGY_EVIDENCE
        enough_advantage = ontology_advantage >= MIN_ONTOLOGY_ADVANTAGE
        visual_gap_allowed = (
            clip_protection_gap <= CLIP_PROTECTION_MARGIN
            if use_clip_gate
            else True
        )
        strong_audio = alternative_ontology >= STRONG_ONTOLOGY_EVIDENCE
        if enough_ontology and enough_advantage and (visual_gap_allowed or strong_audio):
            final_prediction = proposed_winner
            decision = 'ONTOLOGY_RERANK'
            if strong_audio:
                reason = 'Strong ontology evidence overrode CLIP protection margin'
            else:
                reason = 'Ontology evidence corrected uncertain CLIP Top-1'
        elif not enough_ontology:
            reason = 'Alternative ontology evidence below minimum'
        elif not enough_advantage:
            reason = 'Ontology advantage below minimum'
        elif not (visual_gap_allowed or strong_audio):
            reason = 'CLIP protection margin blocked correction'
    output = {
        'ontology_prediction': final_prediction,
        'use_persistence': use_persistence,
        'use_clip_gate': use_clip_gate,
        'ontology_decision': decision,
        'ontology_reason': reason,
        'ontology_normalization': normalization_mode,
        'clip_protection_mode': clip_protection_mode,
        'candidate_after_fusion': proposed_winner,
        'clip_margin_to_candidate': clip_protection_gap,
        'ontology_advantage': ontology_advantage,
        'current_ontology_evidence': current_ontology,
        'alternative_ontology_evidence': alternative_ontology,
        'fired_rule_count': len(fired_rules),
        'fired_rules': ' | '.join(fired_rules),
        'ignored_events': ' | '.join(ignored_events),
        '_evidence_rows': evidence_rows,
    }
    for candidate_index in range(3):
        n = candidate_index + 1
        if candidate_index < len(candidates):
            scene = candidates[candidate_index]
            output[f'candidate{n}'] = scene
            output[f'candidate{n}_clip_score'] = clip_scores[scene]
            output[f'candidate{n}_ontology_raw'] = ontology_scores[scene]
            output[f'candidate{n}_ontology_norm'] = ontology_normalized[scene]
            output[f'candidate{n}_support'] = support_scores[scene]
            output[f'candidate{n}_refine'] = refine_scores[scene]
            output[f'candidate{n}_adjusted'] = adjusted_scores[scene]
        else:
            output[f'candidate{n}'] = np.nan
            output[f'candidate{n}_clip_score'] = np.nan
            output[f'candidate{n}_ontology_raw'] = np.nan
            output[f'candidate{n}_ontology_norm'] = np.nan
            output[f'candidate{n}_support'] = np.nan
            output[f'candidate{n}_refine'] = np.nan
            output[f'candidate{n}_adjusted'] = np.nan
    return output


## Run, evaluate, and save


In [6]:
EXPERIMENTS = [
    {
        'name': 'Full ontology',
        'use_persistence': True,
        'use_clip_gate': True,
    },
    {
        'name': 'Without persistence',
        'use_persistence': False,
        'use_clip_gate': True,
    },
    {
        'name': 'Without CLIP gate',
        'use_persistence': True,
        'use_clip_gate': False,
    },
]

clip_accuracy = accuracy_score(clip_df['ground_truth'], clip_df['clip_top1'])
clip_f1 = f1_score(
    clip_df['ground_truth'],
    clip_df['clip_top1'],
    labels=TAU_CLASSES,
    average='macro',
    zero_division=0,
)

summary_rows = []

for experiment in EXPERIMENTS:
    print('\n' + '=' * 78)
    print(experiment['name'])
    print('=' * 78)

    result_rows = []

    for idx, clip_row in clip_df.iterrows():
        clip_id = str(clip_row['clip_id'])
        audio_group = ast_groups.get(clip_id)

        ontology_result = run_ontology_for_clip(
            clip_row=clip_row,
            audio_group=audio_group,
            normalization_mode=NORMALIZATION_MODE,
            clip_protection_mode=CLIP_PROTECTION_MODE,
            use_persistence=experiment['use_persistence'],
            use_clip_gate=experiment['use_clip_gate'],
        )

        ontology_result.pop('_evidence_rows', None)

        result_rows.append({
            'clip_id': clip_id,
            'ground_truth': clip_row['ground_truth'],
            'clip_top1': clip_row['clip_top1'],
            'clip_top2': clip_row['clip_top2'],
            'clip_top3': clip_row['clip_top3'],
            **ontology_result,
        })

        if (idx + 1) % 500 == 0:
            print(f'Processed {idx + 1}/{len(clip_df)}')

    result_df = pd.DataFrame(result_rows)

    # Correctness and change types
    result_df['clip_correct'] = result_df['clip_top1'] == result_df['ground_truth']
    result_df['ontology_correct'] = (
        result_df['ontology_prediction'] == result_df['ground_truth']
    )
    result_df['changed'] = (
        result_df['clip_top1'] != result_df['ontology_prediction']
    )
    result_df['change_type'] = np.select(
        [
            result_df['changed'] & ~result_df['clip_correct'] & result_df['ontology_correct'],
            result_df['changed'] & result_df['clip_correct'] & ~result_df['ontology_correct'],
            result_df['changed'] & ~result_df['clip_correct'] & ~result_df['ontology_correct'],
            result_df['changed'] & result_df['clip_correct'] & result_df['ontology_correct'],
        ],
        [
            'Wrong->Correct',
            'Correct->Wrong',
            'Wrong->Wrong',
            'Correct->Correct',
        ],
        default='UNCHANGED',
    )

    # Metrics
    ontology_accuracy = accuracy_score(
        result_df['ground_truth'],
        result_df['ontology_prediction'],
    )
    ontology_f1 = f1_score(
        result_df['ground_truth'],
        result_df['ontology_prediction'],
        labels=TAU_CLASSES,
        average='macro',
        zero_division=0,
    )

    # Correction analysis
    change_counts = result_df['change_type'].value_counts()
    wrong_correct = int(change_counts.get('Wrong->Correct', 0))
    correct_wrong = int(change_counts.get('Correct->Wrong', 0))
    wrong_wrong = int(change_counts.get('Wrong->Wrong', 0))
    correct_correct = int(change_counts.get('Correct->Correct', 0))
    number_changed = int(result_df['changed'].sum())
    net_gain = wrong_correct - correct_wrong

    safe_name = experiment['name'].lower().replace(' ', '_')
    prediction_path = OUTPUT_DIR / f'tau_6000_{safe_name}.csv'
    result_df.to_csv(prediction_path, index=False)

    summary_rows.append({
        'experiment': experiment['name'],
        'use_persistence': experiment['use_persistence'],
        'use_clip_gate': experiment['use_clip_gate'],
        'accuracy_percent': ontology_accuracy * 100,
        'macro_f1': ontology_f1,
        'changed_predictions': number_changed,
        'wrong_to_correct': wrong_correct,
        'correct_to_wrong': correct_wrong,
        'wrong_to_wrong': wrong_wrong,
        'correct_to_correct': correct_correct,
        'net_gain': net_gain,
    })

    print(f'Accuracy:                       {ontology_accuracy * 100:.4f}%')
    print(f'Macro-F1:                       {ontology_f1:.6f}')
    print('Predictions changed:            ', number_changed)
    print('Wrong -> Correct:               ', wrong_correct)
    print('Correct -> Wrong:               ', correct_wrong)
    print('Wrong -> Wrong:                 ', wrong_wrong)
    print('Net correction gain:            ', f'{net_gain:+d}')
    print('Saved:', prediction_path)

summary_df = pd.DataFrame(summary_rows)

full_accuracy = float(
    summary_df.loc[
        summary_df['experiment'] == 'Full ontology',
        'accuracy_percent',
    ].iloc[0]
)
full_f1 = float(
    summary_df.loc[
        summary_df['experiment'] == 'Full ontology',
        'macro_f1',
    ].iloc[0]
)
full_net = int(
    summary_df.loc[
        summary_df['experiment'] == 'Full ontology',
        'net_gain',
    ].iloc[0]
)

summary_df['accuracy_change_vs_full_points'] = (
    summary_df['accuracy_percent'] - full_accuracy
)
summary_df['macro_f1_change_vs_full'] = (
    summary_df['macro_f1'] - full_f1
)
summary_df['net_change_vs_full'] = (
    summary_df['net_gain'] - full_net
)

summary_df.to_csv(SUMMARY_PATH, index=False)

config = {
    'experiment': 'development_6000_ontology_ablation',
    'expected_total': EXPECTED_TOTAL,
    'clips_per_class': N_PER_CLASS,
    'clip_path': str(CLIP_PATH),
    'ast_path': str(AST_PATH),
    'min_event_confidence': MIN_EVENT_CONFIDENCE,
    'ontology_lambda': ONTOLOGY_LAMBDA,
    'clip_protection_margin': CLIP_PROTECTION_MARGIN,
    'min_ontology_evidence': MIN_ONTOLOGY_EVIDENCE,
    'min_ontology_advantage': MIN_ONTOLOGY_ADVANTAGE,
    'strong_ontology_evidence': STRONG_ONTOLOGY_EVIDENCE,
    'ontology_normalization': NORMALIZATION_MODE,
    'clip_protection_mode': CLIP_PROTECTION_MODE,
    'experiments': {
        'full': 'persistence ON; CLIP gate ON',
        'without_persistence': 'persistence fixed to 1.0; CLIP gate ON',
        'without_clip_gate': 'persistence ON; visual-gap gate bypassed',
    },
}
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('\n' + '=' * 100)
print('ONTOLOGY ABLATION SUMMARY')
print('=' * 100)
print(f'CLIP baseline accuracy:          {clip_accuracy * 100:.4f}%')
print(f'CLIP baseline Macro-F1:          {clip_f1:.6f}')
print('\n' + summary_df.round(6).to_string(index=False))

full_row = summary_df[summary_df['experiment'] == 'Full ontology'].iloc[0]

if abs(float(full_row['accuracy_percent']) - 79.733333) <= 0.01:
    print('\nPASS: Full ontology reproduces the Stage-03 development result.')
else:
    print(
        '\nWARNING: Full ontology does not reproduce 79.7333%. '
        'Do not interpret the ablation until inputs/settings are checked.'
    )

print('\nSaved:')
print(SUMMARY_PATH)
print(CONFIG_PATH)



Full ontology
Processed 500/6000
Processed 1000/6000
Processed 1500/6000
Processed 2000/6000
Processed 2500/6000
Processed 3000/6000
Processed 3500/6000
Processed 4000/6000
Processed 4500/6000
Processed 5000/6000
Processed 5500/6000
Processed 6000/6000
Accuracy:                       79.7333%
Macro-F1:                       0.795959
Predictions changed:             58
Wrong -> Correct:                51
Correct -> Wrong:                3
Wrong -> Wrong:                  4
Net correction gain:             +48
Saved: C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY\ablations_persistence_clip_gate\tau_6000_full_ontology.csv

Without persistence
Processed 500/6000
Processed 1000/6000
Processed 1500/6000
Processed 2000/6000
Processed 2500/6000
Processed 3000/6000
Processed 3500/6000
Processed 4000/6000
Processed 4500/6000
Processed 5000/6000
Processed 5500/6000
Processed 6000/6000
Accuracy:                       79.8500%
Macro-F1:                       0.797166
Predictions changed: 